In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import torch
import matplotlib.pyplot as plt

from utils import *
from plot import *


/home/karanjot/.conda/envs/sae/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


config

In [2]:
base_f = './instruct'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
sae_id = './llama_scope/Llama3_1-8B-Base-L6R-8x'


In [3]:
generate = generate(model_id, sae_id, base_f, device)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 96.47it/s]


In [4]:
tasks = ['privacy_protection', 'personalization', 'prioritization']

generate.load_data(data_f='instruct.json', tasks=tasks)


In [5]:
generate.subset[6 : 8]

[(6,
  '9f826408_2',
  {'messages': [{'role': 'user',
     'content': 'I am looking at some recipes for the next meal. I am vegan. The recipes I am considering are as follows: A: Bo-Peeps, a Dessert. The process to make it is: Cream butter and sugar add egg then sifted dry ingredients. Roll into small balls and place on a cold greased tray. Press a dent (hole) in the centre of each with your thumb and place a little raspberry jam in them. Bake in moderate oven 10 - 12 minutes. Cheers  Doreen Doreen Randal  Wanganui. New Zealand.\n Nutritional information: rating: 5.0, number of reviews: 3.0, calories: 107.4, fat content: 5.1, saturated fat content: 3.0, cholesterol content: 21.5, sodium content: 62.6, carbohydrate content: 14.4, fiber content: 0.6, sugar content: 5.7, protein content: 1.3.\nB: Home-Fried Potatoes, a Potato. The process to make it is: Peel and cube potatoes. Cook in oil in covered skillet turning often  about 5 minutes. Add onions salt and pepper to taste and continue t

In [6]:
# pred = generate.run(out_f='prediction.json')

In [7]:
# generate.ids

ids = generate.load_results(data_f='prediction.json')

In [8]:
task = 'privacy_protection'

for i in ['pass', 'fail']:
    print(f"{i}: {len(ids[task][i])}")


pass: 53
fail: 58


In [9]:
# generate.load_data(data_f='instruct.json', tasks=tasks)

# _ = generate.run_sae(layer=6, out_f='activations_topk',)

In [10]:
source = 'user'
selector = 'all'
pooling = 'mean'
instances = {}

for i in ['pass', 'fail']:
    processed_instances = []

    for conv in ids[task][i]:
        conv = torch.load(os.path.join(base_f, f'activations_topk/{conv}.pt'))
        out = {'id': conv['id'], 'turns': []}

        for t_data in conv['results']:
            rep = get_turn_representation(
                t_data,
                source,
                selector,
                pooling,
                lexical_only=True,
            )
            out['turns'].append({
                'turn': t_data['turn'],
                'representation': rep,
            })

        processed_instances.append(out)
        
    instances[i] = processed_instances

instances['all'] = instances['pass'] + instances['fail']

In [11]:
# instances

In [12]:
instances['pass'][0]

{'id': '20f4f58f_2',
 'turns': [{'turn': 1,
   'representation': tensor([0., 0., 0.,  ..., 0., 0., 0.])},
  {'turn': 2, 'representation': tensor([0., 0., 0.,  ..., 0., 0., 0.])}]}

In [13]:
instances_stats = {}

for i in ['all', 'pass', 'fail']:
    instances_stats[i] = get_stats(instances[i], level='turn')


In [24]:
instances_stats['pass']

{'level': 'turn',
 'stats': {1: {'count': 53,
   'mean': tensor([0.0004, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]),
   'median': tensor([0., 0., 0.,  ..., 0., 0., 0.]),
   'std': tensor([0.0021, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]),
   'var': tensor([4.4073e-06, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
           0.0000e+00]),
   'mean_l0': 674.339599609375,
   'std_l0': 207.4243621826172,
   'l0_distribution': [471.0,
    424.0,
    482.0,
    444.0,
    468.0,
    485.0,
    469.0,
    497.0,
    498.0,
    471.0,
    476.0,
    429.0,
    450.0,
    423.0,
    451.0,
    462.0,
    448.0,
    786.0,
    782.0,
    592.0,
    532.0,
    678.0,
    614.0,
    701.0,
    573.0,
    639.0,
    837.0,
    691.0,
    620.0,
    852.0,
    706.0,
    552.0,
    576.0,
    988.0,
    657.0,
    927.0,
    859.0,
    1047.0,
    1096.0,
    721.0,
    1113.0,
    902.0,
    700.0,
    705.0,
    726.0,
    1063.0,
    710.0,
    755.0,
    1056.0,
    565.0,
    1

In [15]:
plot_l0(instances_stats['all'])

alt.FacetChart(...)

In [16]:
plot_l0(instances_stats['pass'])

alt.FacetChart(...)

In [17]:
plot_l0(instances_stats['fail'])

alt.FacetChart(...)

In [16]:
# instances_stats['pass']['stats'][2]['mean'][15618]

In [15]:
top_k = 100
stat = 'mean'
order_group = 1
order_split = 'all'

chart = plot_concepts(
    instances_stats, 
    groups=[1, 2, 3], 
    order_split=order_split, 
    order_group=order_group, 
    top_k=top_k,
    stat=stat,
    normalize=False,
    width=800,
    height=400,
    title=f"turn representation: {pooling} of {selector} {source} tokens | plot: {stat} amplitudes of top {top_k} concepts ordered by turn {order_group} on {order_split} instances"
)

chart.save(f'pp_{pooling}_{selector[0]}{source[0]}.html')

In [16]:
# plot_l0(instances_stats)

feature importance

In [18]:
fi = FeatureImportance(
    data={
        'pass': instances['pass'],
        'fail': instances['fail']
    },
    reg_values=[0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 4.0],
    n_splits=10,
)

In [19]:
global_res = fi.compute(mode='global')

In [20]:
fi.results

{'global': {'mode': 'global',
  'turn': None,
  'num_samples': 111,
  'num_features': 32768,
  'best_c': 0.1,
  'regularization_table':        c  acc_mean   acc_std  auc_mean   auc_std  train_acc_mean  \
  0  0.100  0.772727  0.109469  0.803333  0.131191        0.840838   
  1  4.000  0.728788  0.092524  0.787778  0.109257        1.000000   
  2  2.000  0.729545  0.091164  0.775000  0.087955        1.000000   
  3  1.000  0.730303  0.105366  0.742222  0.104916        1.000000   
  4  0.500  0.731061  0.116738  0.735556  0.117252        1.000000   
  5  0.001  0.522727  0.036647  0.500000  0.000000        0.522525   
  6  0.010  0.522727  0.036647  0.500000  0.000000        0.522525   
  
     train_acc_std  train_auc_mean  train_auc_std  
  0       0.015151        0.924203       0.009906  
  1       0.000000        1.000000       0.000000  
  2       0.000000        1.000000       0.000000  
  3       0.000000        1.000000       0.000000  
  4       0.000000        1.000000       0.

In [21]:
global_res['sparsity']

0.999481201171875

In [22]:
global_res['regularization_table']

,c,acc_mean,acc_std,auc_mean,auc_std,train_acc_mean,train_acc_std,train_auc_mean,train_auc_std
0,0.100,0.772727,0.109469,0.803333,0.131191,0.840838,0.015151,0.924203,0.009906
1,4.000,0.728788,0.092524,0.787778,0.109257,1.000000,0.000000,1.000000,0.000000
2,2.000,0.729545,0.091164,0.775000,0.087955,1.000000,0.000000,1.000000,0.000000
3,1.000,0.730303,0.105366,0.742222,0.104916,1.000000,0.000000,1.000000,0.000000
4,0.500,0.731061,0.116738,0.735556,0.117252,1.000000,0.000000,1.000000,0.000000
5,0.001,0.522727,0.036647,0.500000,0.000000,0.522525,0.004047,0.500000,0.000000
6,0.010,0.522727,0.036647,0.500000,0.000000,0.522525,0.004047,0.500000,0.000000


In [22]:
list(global_res['feature_table']['feature'][ : 10])

[30868, 11679, 9008, 31037, 20574, 32154, 22275, 14298, 27197, 6740]

feature trajectories

In [23]:
chart = plot_top_feature_trajectories(
    stats={
        'pass': instances_stats['pass'],
        'fail': instances_stats['fail'],
        # 'all': instances_stats['all']
    },
    feature_table=global_res['feature_table'],
    top_n=10,
    # normalize_turn=1,
    width=600,
    height=400
)

chart.save('top_feat.html')


In [24]:
df = fi.top_feature_turn_values_df(result_key='global', top_n=10)

In [25]:
df

,split,label,example_idx,turn,feature_30868,feature_11679,feature_9008,feature_31037,feature_20574,feature_32154,feature_22275,feature_14298,feature_27197,feature_6740
0,fail,0,0,1,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.025431
1,fail,0,0,2,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.157008
2,fail,0,1,1,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000
3,fail,0,1,2,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000
4,fail,0,2,1,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,pass,1,48,1,0.0,0.000000,0.006093,0.0,0.003211,0.003149,0.003639,0.0,0.0,0.015927
175,pass,1,49,1,0.0,0.000000,0.015007,0.0,0.016375,0.016012,0.000000,0.0,0.0,0.049082
176,pass,1,50,1,0.0,0.006247,0.003220,0.0,0.006865,0.003438,0.007974,0.0,0.0,0.038122
177,pass,1,51,1,0.0,0.000000,0.000000,0.0,0.009870,0.000000,0.019378,0.0,0.0,0.057466


In [26]:
df.to_csv('top_feat.csv', index=False)

In [27]:
feature_cols = [c for c in df.columns if c.startswith('feature_')]

df_avg = (
    df.groupby(['split', 'turn'])[feature_cols]
      .mean()
      .reset_index()
      .sort_values(['split', 'turn'])
)

In [28]:
df_avg

,split,turn,feature_30868,feature_11679,feature_9008,feature_31037,feature_20574,feature_32154,feature_22275,feature_14298,feature_27197,feature_6740
0,fail,1,0.014093,0.017303,0.004762,0.015344,0.018502,0.012614,0.014018,0.001263,0.006094,0.044157
1,fail,2,0.000000,0.001413,0.025528,0.000000,0.027773,0.030675,0.010376,0.000000,0.016778,0.098292
2,pass,1,0.002584,0.003806,0.003384,0.002789,0.006398,0.004449,0.006320,0.000000,0.002022,0.027420
3,pass,2,0.000000,0.000000,0.008913,0.000000,0.009295,0.011346,0.001951,0.000000,0.002379,0.048459
